<a href="https://colab.research.google.com/github/angelrecalde2024/Power-System-Planning-and-Transmission-Design-2026/blob/main/INGP1118_TransmissionExpansion_TEP_DC_OPF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TRANSMISSION EXPANSION PLANNING based on DC-OPF

We start from the DC-OPF where the objective is the minimization of the total cost of generation, and the constraints are: decoupled DC power balance on each bus, generation limits, slack angle, line limits.

So, the network topology was fixed, line susceptance matrix B was constant, and the decision variables are generation powers and bus angles.

What changes when Transmission Expansion Planning is introduced? The network topology becomes endogenous, some lineas are optional, the power flow equations depend on investment decisions. This turns the problem into a Mixed Integer Linear Program (MILP).

The core concept of this problem is the binary line investment decision. For each candidate line,

$$
y_{\ell} \in\{0,1\}
$$

When $y_{\ell}$ is 0, the line is not built, when it is 1, the line is built.

The TEP DC-OPF object is now,

$$
\min \sum_g C_g\left(P_g\right)+\sum_{\ell \in \text { cand }} C_{\ell}^{i n v} y_{\ell}
$$

Where the cost of investment is added to the generation cost.

Where is the modelling difficulty? We know that the DC power flow on a candidate line is,

$$
F_{\ell}=\frac{1}{X_{\ell}}\left(\theta_i-\theta_j\right)
$$

But this equation should only exist if $y_{\ell}$ is 1. If it is 0, the flow must be 0, and the line must not influence the B matrix. This creates a bilinear term if written naively,

$$
F_{\ell}=y_{\ell}\frac{1}{X_{\ell}}\left(\theta_i-\theta_j\right)
$$

Which is nonlinear because it holds a product of binary and continuous. To overcome this nonlinearity, we could use the standard linear formulation (Big-M method). Instead of modifying B directly, we define a line flow variable $F_{\ell}$ that enforces
$
F_{\ell}=\frac{1}{X_{\ell}}\left(\theta_i-\theta_j\right)
$
for existing lines. For candidate lines, we apply the following relaxation with Big-M:

$$
-M\left(1-y_{\ell}\right) \leq F_{\ell}-\frac{1}{X_{\ell}}\left(\theta_i-\theta_j\right) \leq M\left(1-y_{\ell}\right)
$$

If $y_{\ell}$ is 0, $F_{\ell}$ is 0. On the contrary, if $y_{\ell}$ is 1, $F_{\ell}$ goes to normal limits.

This is the classical DC-TEP MILP (Mixed Integer Linear Programming) formulation. This way is the standard LP compatible way.

This is not the only way to model the problem. Other formulations include,


*   Disjunctive Programming (requires transformation to MILP with Big-M or convex hull reformulation).
*   PTDF formulation. Instead of angle equations we use Power Transfer Distribution Factor with $F=PTDF*P$. But PTDF depends on topology, so, it becomes complicated when topology changes. It is used mostly in large-scale TEP with decomposition.
*   Benders decomposition. It is common in large TEP. The master problem is to choose lines with binary variables. The suproblem is the DC-OPF. There are a number of cuts that are generated iteratively. This method is more scalable, but conceptually more advanced.

FORMAL MATHEMATICAL FORM

The variables are: $P_g$ (generation power), $θ_i$ (bus angles), $F_l$ (power flow in transmission lines), $y_l$ (binary variable). The objevtive is as follows,



OPTIONS FOR EXPANSION

There are expansion of existing right-of-way, or expansion of new right-of-way.

$$
\min \sum_g C_g\left(P_g\right)+\sum_{\ell \in \text { cand }} C_{\ell}^{i n v} y_{\ell}
$$

Subject to the DC nodal balance,

$$
\sum_g P_g-P_d=\sum_{\ell} A_{i \ell} F_{\ell}
$$

generation limits,

$$
P_g^{\min } \leq P_g \leq P_g^{\max }
$$

slack angle,

$$
θ_{slack} = 0
$$

Existing line flow equations,

$$
F_{\ell}=\frac{1}{X_{\ell}}\left(\theta_i-\theta_j\right)
$$

Candidate line Big-M linking,

$$
-M\left(1-y_{\ell}\right) \leq F_{\ell}-\frac{1}{X_{\ell}}\left(\theta_i-\theta_j\right) \leq M\left(1-y_{\ell}\right)
$$

Line capacity conditional,

$$
-y_{\ell} F_{\ell}^{\max } \leq F_{\ell} \leq y_{\ell} F_{\ell}^{\max }
$$

We also have a 'allow N circuits per corridor' constraint. There is a knob that the user can use as a grouping constraint by corridor.

$$
\sum_{\ell \in \mathcal{C}(i, j)} y_{\ell} \leq N_{\max }
$$

where $\mathcal{C}(i, j)$ is the set of candidate rows that share the same corridor endpoints $(i, j)$. $N_{\max }$ is the knob to allow the number of corridors in parallel.






In [3]:
# ============================================================
# DC-TEP (single-period) MILP with Pyomo + HiGHS
# - Existing lines always on
# - Candidate lines selected by binaries y
# - Piecewise-linear gen cost approximation of a+bP+cP^2
# - Strict feasibility (no ENS)
# - Up to N parallel candidate circuits per corridor (knob)
# ============================================================

!pip -q install pyomo highspy openpyxl pandas numpy

import math
import numpy as np
import pandas as pd
import pyomo.environ as pyo

# -----------------------
# USER KNOBS
# -----------------------
EXCEL = "/content/Data5busSystemTEP.xlsx"   # Colab: "/content/Data5busSystemTEP.xlsx"
BASE_MVA = 100.0
SLACK_BUS_EXT = 1

K_SEG = 5  # piecewise segments for generator cost
MAX_PARALLEL_PER_CORRIDOR = 2  # <= 2 candidates per (from,to) corridor

# Angle bounds for Big-M strength (radians)
THETA_MAX = math.pi  # [-pi, pi] is a safe teaching default

# Big-M safety multiplier (keep modest; too huge makes numerics worse)
M_MULT = 1.2

LOAD_MULT = 3.00   # e.g., 1.25 means +25% demand everywhere

# -----------------------
# READ EXCEL (validate headers)
# -----------------------
gen_df  = pd.read_excel(EXCEL, sheet_name="generator")
load_df = pd.read_excel(EXCEL, sheet_name="load")
line_df = pd.read_excel(EXCEL, sheet_name="lines")
cand_df = pd.read_excel(EXCEL, sheet_name="candidates")

# Required columns (exact)
req_gen  = {"generator bus","a","b","c","Pmin","Pmax"}
req_load = {"load bus","Pload"}
req_line = {"from","to","X","Flow limit"}
req_cand = {"from","to","X","Flow limit","Investment cost"}

if not req_gen.issubset(gen_df.columns):
    raise ValueError(f"'generator' missing columns. Found: {list(gen_df.columns)}")
if not req_load.issubset(load_df.columns):
    raise ValueError(f"'load' missing columns. Found: {list(load_df.columns)}")
if not req_line.issubset(line_df.columns):
    raise ValueError(f"'lines' missing columns. Found: {list(line_df.columns)}")
if not req_cand.issubset(cand_df.columns):
    raise ValueError(
        "'candidates' missing columns. "
        f"Required: {sorted(req_cand)} ; Found: {list(cand_df.columns)}"
    )

# -----------------------
# Build external bus set (allow non-continuous numbering)
# Include candidates endpoints too
# -----------------------
buses_ext = sorted(
    set(gen_df["generator bus"].astype(int))
    | set(load_df["load bus"].astype(int))
    | set(line_df["from"].astype(int))
    | set(line_df["to"].astype(int))
    | set(cand_df["from"].astype(int))
    | set(cand_df["to"].astype(int))
)
if SLACK_BUS_EXT not in buses_ext:
    raise ValueError(f"Slack bus {SLACK_BUS_EXT} not in buses: {buses_ext}")

ext2int = {b:i for i,b in enumerate(buses_ext)}
int2ext = {i:b for b,i in ext2int.items()}
nb = len(buses_ext)
slack_i = ext2int[SLACK_BUS_EXT]

# -----------------------
# Load (MW) per internal bus
# -----------------------
Pd = np.zeros(nb, dtype=float)
for _, r in load_df.iterrows():
    Pd[ext2int[int(r["load bus"])]] += float(r["Pload"])

Pd = LOAD_MULT * Pd

# -----------------------
# Generators
# -----------------------
ng = len(gen_df)
gen_bus = np.zeros(ng, dtype=int)
Pmin = np.zeros(ng, dtype=float)
Pmax = np.zeros(ng, dtype=float)
a = np.zeros(ng, dtype=float)
b = np.zeros(ng, dtype=float)
c = np.zeros(ng, dtype=float)

for g, (_, r) in enumerate(gen_df.iterrows()):
    gen_bus[g] = ext2int[int(r["generator bus"])]
    a[g] = float(r["a"]); b[g] = float(r["b"]); c[g] = float(r["c"])
    Pmin[g] = float(r["Pmin"]); Pmax[g] = float(r["Pmax"])

if Pmax.sum() + 1e-6 < Pd.sum():
    raise ValueError(
        f"Insufficient generation capacity: sum Pmax={Pmax.sum():.3f} < total load={Pd.sum():.3f}"
    )

gens_at_bus = {i: [] for i in range(nb)}
for g in range(ng):
    gens_at_bus[int(gen_bus[g])].append(g)

# -----------------------
# Lines: build unified branch list
# We'll keep existing lines and candidate lines in separate index sets.
# Each branch stores (from_i, to_i, X, Fmax, type, inv_cost, corridor_key)
# corridor_key is undirected (min,max) for parallel grouping.
# -----------------------
existing = []
for _, r in line_df.iterrows():
    fi = ext2int[int(r["from"])]
    ti = ext2int[int(r["to"])]
    X  = float(r["X"])
    Fm = float(r["Flow limit"])
    if abs(X) < 1e-9:
        raise ValueError("Found near-zero X in 'lines' sheet.")
    existing.append((fi, ti, X, Fm))

candidates = []
for idx, r in cand_df.iterrows():
    fi = ext2int[int(r["from"])]
    ti = ext2int[int(r["to"])]
    X  = float(r["X"])
    Fm = float(r["Flow limit"])
    inv = float(r["Investment cost"])
    if abs(X) < 1e-9:
        raise ValueError("Found near-zero X in 'candidates' sheet.")
    if inv < 0:
        raise ValueError("Investment cost must be nonnegative.")
    corridor = (min(fi, ti), max(fi, ti))
    candidates.append((fi, ti, X, Fm, inv, corridor))

nl = len(existing)
nc = len(candidates)

print("=== Data summary ===")
print(f"Buses: {nb}  Generators: {ng}  Existing lines: {nl}  Candidate lines: {nc}")
print(f"Total load: {Pd.sum():.3f} MW  Sum Pmax: {Pmax.sum():.3f} MW")
print(f"Slack bus ext={SLACK_BUS_EXT} (int={slack_i})")
print(f"MAX_PARALLEL_PER_CORRIDOR={MAX_PARALLEL_PER_CORRIDOR}")

# -----------------------
# Piecewise-linear generator cost:
# C(P)=a+bP+cP^2 approximated by K segments from Pmin..Pmax
# We'll omit constants in objective (safe).
# -----------------------
def pwl_params(Pmin_g, Pmax_g, ag, bg, cg, K):
    pts = np.linspace(Pmin_g, Pmax_g, K+1)
    dP = np.diff(pts)
    C = ag + bg*pts + cg*(pts**2)
    slope = np.diff(C) / dP  # $/MW segment slope
    return dP, slope

dP_seg = np.zeros((ng, K_SEG), dtype=float)
slope_seg = np.zeros((ng, K_SEG), dtype=float)
for g in range(ng):
    dP_seg[g,:], slope_seg[g,:] = pwl_params(Pmin[g], Pmax[g], a[g], b[g], c[g], K_SEG)

# -----------------------
# Big-M for candidate DC relation
# DC flow: F = BASE_MVA*(theta_f - theta_t)/X
# If y=0, relax equality by +-M.
# Use theta bounds => max |theta_f - theta_t| <= 2*THETA_MAX
# Then max |BASE_MVA*(dtheta)/X| <= BASE_MVA*(2*THETA_MAX)/|X|
# Choose M >= that + Fmax (conservative).
# -----------------------
def bigM_for_candidate(X, Fmax):
    return M_MULT * (abs(Fmax) + BASE_MVA*(2*THETA_MAX)/abs(X))

# -----------------------
# MODEL (MILP)
# -----------------------
m = pyo.ConcreteModel()

m.B = pyo.RangeSet(0, nb-1)
m.G = pyo.RangeSet(0, ng-1)
m.K = pyo.RangeSet(0, K_SEG-1)
m.LE = pyo.RangeSet(0, nl-1)     # existing lines
m.LC = pyo.RangeSet(0, nc-1)     # candidate lines

# Variables
m.theta = pyo.Var(m.B, bounds=(-THETA_MAX, THETA_MAX))
m.theta[slack_i].fix(0.0)

# PWL segment vars
m.xseg = pyo.Var(m.G, m.K, domain=pyo.NonNegativeReals)

def seg_ub(mdl, g, k):
    return mdl.xseg[g, k] <= float(dP_seg[g, k])
m.seg_ub = pyo.Constraint(m.G, m.K, rule=seg_ub)

# Generator output (MW)
def Pg_expr(mdl, g):
    return float(Pmin[g]) + sum(mdl.xseg[g, k] for k in mdl.K)
m.Pg = pyo.Expression(m.G, rule=Pg_expr)

# Existing line flows (MW)
m.F_e = pyo.Var(m.LE)

# Candidate line flows (MW) + binaries
m.F_c = pyo.Var(m.LC)
m.y = pyo.Var(m.LC, domain=pyo.Binary)

# -----------------------
# Existing line DC relation + limits
# -----------------------
def existing_dc(mdl, l):
    fi, ti, X, Fm = existing[l]
    return mdl.F_e[l] == BASE_MVA*(mdl.theta[fi] - mdl.theta[ti]) / float(X)
m.ex_dc = pyo.Constraint(m.LE, rule=existing_dc)

def existing_lim_ub(mdl, l):
    _, _, _, Fm = existing[l]
    return mdl.F_e[l] <= float(Fm)
def existing_lim_lb(mdl, l):
    _, _, _, Fm = existing[l]
    return -mdl.F_e[l] <= float(Fm)
m.ex_ub = pyo.Constraint(m.LE, rule=existing_lim_ub)
m.ex_lb = pyo.Constraint(m.LE, rule=existing_lim_lb)

# -----------------------
# Candidate line on/off DC relation + conditional limits
# -----------------------
def cand_dc_ub(mdl, l):
    fi, ti, X, Fm, inv, corridor = candidates[l]
    M = bigM_for_candidate(X, Fm)
    # F - BASE_MVA*(dtheta)/X <= M*(1-y)
    return mdl.F_c[l] - BASE_MVA*(mdl.theta[fi] - mdl.theta[ti]) / float(X) <= M*(1 - mdl.y[l])

def cand_dc_lb(mdl, l):
    fi, ti, X, Fm, inv, corridor = candidates[l]
    M = bigM_for_candidate(X, Fm)
    # -(F - BASE_MVA*(dtheta)/X) <= M*(1-y)
    return -(mdl.F_c[l] - BASE_MVA*(mdl.theta[fi] - mdl.theta[ti]) / float(X)) <= M*(1 - mdl.y[l])

m.cand_dc_ub = pyo.Constraint(m.LC, rule=cand_dc_ub)
m.cand_dc_lb = pyo.Constraint(m.LC, rule=cand_dc_lb)

def cand_lim_ub(mdl, l):
    _, _, _, Fm, _, _ = candidates[l]
    return mdl.F_c[l] <= float(Fm) * mdl.y[l]

def cand_lim_lb(mdl, l):
    _, _, _, Fm, _, _ = candidates[l]
    return -mdl.F_c[l] <= float(Fm) * mdl.y[l]

m.cand_ub = pyo.Constraint(m.LC, rule=cand_lim_ub)
m.cand_lb = pyo.Constraint(m.LC, rule=cand_lim_lb)

# -----------------------
# Corridor parallel limit: sum y <= MAX_PARALLEL_PER_CORRIDOR for each corridor
# -----------------------
corridors = {}
for l in range(nc):
    corridors.setdefault(candidates[l][5], []).append(l)

m.corr_set = pyo.Set(initialize=list(corridors.keys()), dimen=2)

def corr_limit(mdl, i, j):
    ls = corridors[(i, j)]
    return sum(mdl.y[l] for l in ls) <= int(MAX_PARALLEL_PER_CORRIDOR)

m.corr_limit = pyo.Constraint(m.corr_set, rule=corr_limit)

# -----------------------
# Power balance at each bus using flow incidence
# net injection = net outflow (existing + candidate)
# sum Pg_at_bus - Pd == sum_out F - sum_in F
# -----------------------
# Precompute incidence lists
out_e = {i: [] for i in range(nb)}
in_e  = {i: [] for i in range(nb)}
for l in range(nl):
    fi, ti, _, _ = existing[l]
    out_e[fi].append(l)
    in_e[ti].append(l)

out_c = {i: [] for i in range(nb)}
in_c  = {i: [] for i in range(nb)}
for l in range(nc):
    fi, ti, _, _, _, _ = candidates[l]
    out_c[fi].append(l)
    in_c[ti].append(l)

def balance(mdl, i):
    Pg_i = sum(mdl.Pg[g] for g in gens_at_bus[i]) if gens_at_bus[i] else 0.0
    net_out = sum(mdl.F_e[l] for l in out_e[i]) - sum(mdl.F_e[l] for l in in_e[i])
    net_out += sum(mdl.F_c[l] for l in out_c[i]) - sum(mdl.F_c[l] for l in in_c[i])
    return Pg_i - float(Pd[i]) == net_out

m.balance = pyo.Constraint(m.B, rule=balance)

# -----------------------
# Objective: PWL gen cost + investment cost
# (PWL cost uses slopes * segment MW; constants omitted)
# -----------------------
gen_cost = sum(float(slope_seg[g, k]) * m.xseg[g, k] for g in range(ng) for k in range(K_SEG))
inv_cost = sum(float(candidates[l][4]) * m.y[l] for l in range(nc))
m.obj = pyo.Objective(expr=gen_cost + inv_cost, sense=pyo.minimize)

# -----------------------
# Solve
# -----------------------
solver = pyo.SolverFactory("highs")
res = solver.solve(m, tee=False, load_solutions=False)
term = str(res.solver.termination_condition).lower()
print("Solver:", res.solver.status, res.solver.termination_condition)
if "optimal" not in term and "feasible" not in term:
    raise RuntimeError("TEP MILP not solved (infeasible or solver error). Check data / limits.")
m.solutions.load_from(res)

# ============================================================
# REPORTING
# ============================================================

# Generator dispatch
Pg = np.array([pyo.value(m.Pg[g]) for g in range(ng)], dtype=float)
gen_rows = []
quad_cost_total = 0.0
for g in range(ng):
    bus_ext = int2ext[int(gen_bus[g])]
    P = Pg[g]
    Cq = float(a[g] + b[g]*P + c[g]*P*P)
    quad_cost_total += Cq
    gen_rows.append([g, bus_ext, P, Cq])

gen_table = pd.DataFrame(gen_rows, columns=["Gen", "Bus", "Pg (MW)", "Quadratic cost a+bP+cP^2 ($/h)"])
print("\n=== Generator dispatch ===")
display(gen_table)
print(f"Total quadratic generation cost (reported) = {quad_cost_total:.3f} $/h")

# Candidate build decisions
cand_rows = []
inv_total = 0.0
for l in range(nc):
    fi, ti, X, Fm, inv, corridor = candidates[l]
    y = int(round(pyo.value(m.y[l])))
    F = float(pyo.value(m.F_c[l]))
    inv_total += inv*y
    cand_rows.append([
        l,
        int2ext[fi], int2ext[ti],
        X, Fm, inv, y, F
    ])

cand_table = pd.DataFrame(
    cand_rows,
    columns=["CandID","From","To","X (pu)","Limit (MW)","Investment cost","Build y","Flow (MW)"]
)
print("\n=== Candidate line decisions ===")
display(cand_table.sort_values(["Build y","CandID"], ascending=[False, True]))
print(f"Total investment cost = {inv_total:.3f}")

# Existing line flows + limit flags
ex_rows = []
tol = 1e-6
for l in range(nl):
    fi, ti, X, Fm = existing[l]
    F = float(pyo.value(m.F_e[l]))
    within = "YES" if abs(F) <= Fm + tol else "NO"
    ex_rows.append([l, int2ext[fi], int2ext[ti], X, Fm, F, within])

ex_table = pd.DataFrame(ex_rows, columns=["Line","From","To","X (pu)","Limit (MW)","Flow (MW)","Within?"])
print("\n=== Existing line flows ===")
display(ex_table)

# Objective decomposition (PWL part is not directly quadratic; show both)
print("\n=== Objective components ===")
print(f"PWL generation objective part (no constants) = {pyo.value(gen_cost):.6f}")
print(f"Investment objective part = {pyo.value(inv_cost):.6f}")
print(f"Total objective = {pyo.value(m.obj):.6f}")

# Corridor utilization (how many built per corridor)
corr_rows = []
for (i, j), ls in corridors.items():
    built = sum(int(round(pyo.value(m.y[l]))) for l in ls)
    corr_rows.append([int2ext[i], int2ext[j], built, MAX_PARALLEL_PER_CORRIDOR, ls])
corr_table = pd.DataFrame(corr_rows, columns=["Corridor bus i","Corridor bus j","Built circuits","Max allowed","Candidate IDs"])
print("\n=== Corridor build counts ===")
display(corr_table.sort_values(["Built circuits"], ascending=False))

=== Data summary ===
Buses: 5  Generators: 2  Existing lines: 4  Candidate lines: 5
Total load: 630.000 MW  Sum Pmax: 1400.000 MW
Slack bus ext=1 (int=0)
MAX_PARALLEL_PER_CORRIDOR=2
Solver: ok optimal

=== Generator dispatch ===


,Gen,Bus,Pg (MW),Quadratic cost a+bP+cP^2 ($/h)
0,0,1,280.0,36808.5200
1,1,3,350.0,1969.5125


Total quadratic generation cost (reported) = 38778.032 $/h

=== Candidate line decisions ===


,CandID,From,To,X (pu),Limit (MW),Investment cost,Build y,Flow (MW)
1,1,1,5,0.2112,175.0,8800000.0,1,105.0
3,3,3,4,0.0845,175.0,3500000.0,1,175.0
0,0,1,2,0.0845,175.0,3500000.0,0,0.0
2,2,2,4,0.2112,175.0,8800000.0,0,0.0
4,4,2,3,0.0845,175.0,5000000.0,0,-0.0


Total investment cost = 12300000.000

=== Existing line flows ===


,Line,From,To,X (pu),Limit (MW),Flow (MW),Within?
0,0,1,2,0.0845,175.0,70.0,YES
1,1,1,5,0.2112,175.0,105.0,YES
2,2,2,4,0.2112,175.0,-140.0,YES
3,3,3,4,0.0845,175.0,175.0,YES



=== Objective components ===
PWL generation objective part (no constants) = 31283.372167
Investment objective part = 12300000.000000
Total objective = 12331283.372167

=== Corridor build counts ===


,Corridor bus i,Corridor bus j,Built circuits,Max allowed,Candidate IDs
1,1,5,1,2,[1]
3,3,4,1,2,[3]
0,1,2,0,2,[0]
2,2,4,0,2,[2]
4,2,3,0,2,[4]
